# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}\n")

## 2. Data Overview
Review available record sets, their `@id`s, and the fields/columns included.

The dataset may contain multiple tables ("record sets") and each column/field can be uniquely referenced by its `@id`.

In [ ]:
# Discover available record sets
print('Available Record Sets:')
for rs in metadata.record_sets:
    rs_id = getattr(rs, '@id', None)
    rs_name = getattr(rs, 'name', None)
    print(f'  @id: {rs_id}, name: {rs_name}')
    print('    Fields/Columns:')
    for field in getattr(rs, 'fields', []):
        field_id = getattr(field, '@id', None)
        field_name = getattr(field, 'name', None)
        print(f'      @id: {field_id}, name: {field_name}')

## 3. Data Extraction
Load data from the primary record set into a DataFrame.

**NOTE:** Use the record set and field `@id`s from the overview above. If the dataset has multiple record sets, you can extract each into a separate DataFrame.

In [ ]:
# Collect all record set @ids
record_sets_ids = [getattr(rs, '@id', None) for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    print(f'Loading records for record set: {record_set_id}')
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f'Columns in record set {record_set_id}:', dataframes[record_set_id].columns.tolist())
    print(dataframes[record_set_id].head(), '\n')

# For the rest of this notebook, we'll focus on the primary protein record set.
# Replace with the main record set @id for proteins (typically ends with something like 'protein' or similar)
protein_record_set_id = None
for rs in metadata.record_sets:
    if 'protein' in getattr(rs, 'name', '').lower() or 'protein' in getattr(rs, '@id', '').lower():
        protein_record_set_id = getattr(rs, '@id')
        break

# Fallback if not found by name, pick the first
if protein_record_set_id is None and record_sets_ids:
    protein_record_set_id = record_sets_ids[0]

print(f'Primary protein record set selected: {protein_record_set_id}')
protein_df = dataframes[protein_record_set_id]
print(protein_df.columns.tolist())
protein_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic EDA: filtering, normalization, and grouping.

We will select a numeric field such as 'Abundance' or 'Peptide_Count', referenced by their column `@id`. Consult the column list printed above.

In [ ]:
# Choose a numeric field @id (replace with the real @id if your schema is different)
# This is an example; use e.g., 'abundance', 'cr:Abundance' or similar from previous cell output
numeric_field_candidates = [col for col in protein_df.columns if any(token in col.lower() for token in ['abundance', 'count', 'mw', 'number', 'coverage', 'intensity'])]
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]
else:
    numeric_field_id = protein_df.columns[0]  # fallback

print(f"Using numeric field: {numeric_field_id}")

# Convert to numeric, handling non-numeric gracefully
protein_df[numeric_field_id] = pd.to_numeric(protein_df[numeric_field_id], errors='coerce')
threshold = protein_df[numeric_field_id].quantile(0.75)
filtered_df = protein_df[protein_df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt grouping by a categorical field (e.g., 'Gene' or 'Description')
group_field_candidates = [col for col in protein_df.columns if any(token in col.lower() for token in ['gene', 'description', 'accession', 'group', 'sample', 'type'])]
if group_field_candidates:
    group_field = group_field_candidates[0]
else:
    group_field = protein_df.columns[0]  # fallback

print(f"Grouping by: {group_field}")
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and explore relationships.

We'll use boxplots and scatter plots to visualize filtered and normalized data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Boxplot of the numeric field
plt.figure(figsize=(8, 5))
sns.boxplot(y=protein_df[numeric_field_id].dropna())
plt.title(f'Distribution of {numeric_field_id}')
plt.ylabel(numeric_field_id)
plt.show()

# Histogram of normalized field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[f'{numeric_field_id}_normalized'].dropna(), kde=True)
plt.title(f'Normalized {numeric_field_id} (filtered records)')
plt.xlabel(f'{numeric_field_id}_normalized')
plt.show()

# If group_field is categorical, plot group means
if group_field in filtered_df.columns and filtered_df[group_field].nunique() < 20:
    plt.figure(figsize=(10, 5))
    sns.barplot(
        data=grouped_df,
        x=group_field, y=numeric_field_id
    )
    plt.title(f'Average {numeric_field_id} by {group_field} (filtered)')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
We loaded and explored the FAIR^2 dataset using `mlcroissant`, reviewed its record sets and schema, and conducted basic data processing/visualization on protein quantification data.

- Leverage unique `@id` references for reproducible data handling.
- For deeper analysis (domain-specific filtering, cross-record set joins, or advanced normalization), inspect the field taxonomy further using record set and field lists.
- This notebook provides a generalizable framework for future Croissant-formatted datasets.